
# Ddri 공식 API 검증 노트북

- 목적: `OA-15493 서울시 공공자전거 따릉이 실시간 대여정보`만 기준으로 서비스 연동 가능성을 검증한다.
- 범위: 비공식 `bikeseoul` 검증과 `OA-21285` 보조 API 검토는 이번 노트북에서 제외한다.
- 참조: `works/00_overview/08_ddri_service_output_logic_draft.md`



## 검증 플랜

이번 노트북에서 확인할 항목:

1. 공식 API의 실제 서비스명과 기본 호출 형식이 명세서와 일치하는가
2. `1000건` 단위 페이지네이션이 실제로 어떻게 동작하는가
3. 응답 스키마가 서비스 입력(`station_id`, 현재 자전거 수, capacity, 좌표)으로 충분한가
4. 대표 15개 스테이션이 공식 API와 얼마나 매칭되는가
5. 현재 서비스 범위인 전체 161개 스테이션이 공식 API와 얼마나 매칭되는가
6. 명세서에 적힌 `stationId` 선택 파라미터가 실제 필터로 동작하는가
7. 잔여 자전거 계산 연동을 위해 어떤 매핑 테이블과 후처리가 필요한가


In [1]:

from pathlib import Path
import os
import math
import json

import pandas as pd
import requests

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
CHENG80_DIR = BASE_DIR / 'cheng80'
OUTPUT_DIR = CHENG80_DIR / 'api_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def load_api_keys():
    env_file = CHENG80_DIR / 'api_keys.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                os.environ[k.strip()] = v.strip()
    return {
        'bike': os.environ.get('SEOUL_BIKE_API_KEY'),
    }


API_KEYS = load_api_keys()
SEOUL_BIKE_KEY = API_KEYS.get('bike')

print(f'BASE_DIR: {BASE_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'따릉이 API 키: {"로드됨" if SEOUL_BIKE_KEY else "없음"}')


BASE_DIR: /Users/cheng80/Desktop/ddri_work
OUTPUT_DIR: /Users/cheng80/Desktop/ddri_work/cheng80/api_output
따릉이 API 키: 로드됨



## 1. 공식 API 기본 형식 확인

명세서 기준 핵심 정보:

- 서비스명: `bikeList`
- 기본 URL 형식: `http://openapi.seoul.go.kr:8088/{KEY}/json/bikeList/{START_INDEX}/{END_INDEX}/`
- 1회 최대 조회 건수: `1000`
- 주요 출력 필드: `rackTotCnt`, `parkingBikeTotCnt`, `shared`, `stationLatitude`, `stationLongitude`, `stationId`, `stationName`


In [2]:

def fetch_bike_list(start: int, end: int, api_key: str | None = None):
    key = api_key or SEOUL_BIKE_KEY
    if not key:
        raise ValueError('따릉이 API 키가 필요합니다. cheng80/api_keys.env 확인.')

    url = f'http://openapi.seoul.go.kr:8088/{key}/json/bikeList/{start}/{end}/'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    payload = data.get('rentBikeStatus', {})
    rows = payload.get('row', [])
    result = payload.get('RESULT', data.get('RESULT', {}))
    return {
        'url': url,
        'data': data,
        'rows': rows,
        'result': result,
        'list_total_count': payload.get('list_total_count'),
    }


sample = fetch_bike_list(1, 5)
print('샘플 URL:', sample['url'])
print('응답 최상위 키:', list(sample['data'].keys()))
print('RESULT:', sample['result'])
print('list_total_count:', sample['list_total_count'])
print('row 수:', len(sample['rows']))
print('첫 row 샘플:')
print(json.dumps(sample['rows'][0], ensure_ascii=False, indent=2))


샘플 URL: http://openapi.seoul.go.kr:8088/6776635665636865343076784d5562/json/bikeList/1/5/
응답 최상위 키: ['rentBikeStatus']
RESULT: {'CODE': 'INFO-000', 'MESSAGE': '정상 처리되었습니다.'}
list_total_count: 5
row 수: 5
첫 row 샘플:
{
  "rackTotCnt": "15",
  "stationName": "102. 망원역 1번출구 앞",
  "parkingBikeTotCnt": "23",
  "shared": "153",
  "stationLatitude": "37.55564880",
  "stationLongitude": "126.91062927",
  "stationId": "ST-4"
}


## 2. 페이지네이션 검증

In [3]:

page_checks = []
all_rows = []

for start in [1, 1001, 2001, 3001, 4001]:
    end = start + 999
    fetched = fetch_bike_list(start, end)
    rows = fetched['rows']
    result = fetched['result']
    page_checks.append({
        'start_index': start,
        'end_index': end,
        'result_code': result.get('CODE'),
        'result_message': result.get('MESSAGE'),
        'list_total_count': fetched['list_total_count'],
        'returned_rows': len(rows),
    })
    if rows:
        all_rows.extend(rows)

page_summary_df = pd.DataFrame(page_checks)
page_summary_df.to_csv(OUTPUT_DIR / 'ddri_seoul_bike_page_summary.csv', index=False, encoding='utf-8-sig')

api_df = pd.DataFrame(all_rows).drop_duplicates().reset_index(drop=True)
api_df['station_num'] = api_df['stationName'].astype(str).str.extract(r'^(\d+)\.')[0]
api_df.to_csv(OUTPUT_DIR / 'ddri_seoul_bike_realtime_all_pages.csv', index=False, encoding='utf-8-sig')

print(page_summary_df.to_string(index=False))
print()
print('통합 row 수:', len(api_df))
print('고유 stationId 수:', api_df['stationId'].nunique())
print('저장:', OUTPUT_DIR / 'ddri_seoul_bike_page_summary.csv')
print('저장:', OUTPUT_DIR / 'ddri_seoul_bike_realtime_all_pages.csv')


 start_index  end_index result_code result_message  list_total_count  returned_rows
           1       1000    INFO-000    정상 처리되었습니다.            1000.0           1000
        1001       2000    INFO-000    정상 처리되었습니다.            1000.0           1000
        2001       3000    INFO-000    정상 처리되었습니다.             684.0            684
        3001       4000        None           None               NaN              0
        4001       5000        None           None               NaN              0

통합 row 수: 2684
고유 stationId 수: 2684
저장: /Users/cheng80/Desktop/ddri_work/cheng80/api_output/ddri_seoul_bike_page_summary.csv
저장: /Users/cheng80/Desktop/ddri_work/cheng80/api_output/ddri_seoul_bike_realtime_all_pages.csv


## 3. `stationId` 선택 파라미터 실동작 검증

In [4]:

test_station_num = '2312'
test_station_api_id = 'ST-795'
filter_test_urls = [
    f'http://openapi.seoul.go.kr:8088/{SEOUL_BIKE_KEY}/json/bikeList/1/5/{test_station_num}',
    f'http://openapi.seoul.go.kr:8088/{SEOUL_BIKE_KEY}/json/bikeList/1/5/stationId/{test_station_num}',
    f'http://openapi.seoul.go.kr:8088/{SEOUL_BIKE_KEY}/json/bikeList/1/5/?stationId={test_station_num}',
    f'http://openapi.seoul.go.kr:8088/{SEOUL_BIKE_KEY}/json/bikeList/1/5/?stationId={test_station_api_id}',
]

filter_rows = []
for url in filter_test_urls:
    r = requests.get(url, timeout=30)
    try:
        body = r.json()
    except Exception:
        body = {'raw_text': r.text[:500]}
    if 'rentBikeStatus' in body:
        returned_rows = len(body['rentBikeStatus'].get('row', []))
        first_station = body['rentBikeStatus'].get('row', [{}])[0].get('stationName') if returned_rows else None
        code = body['rentBikeStatus'].get('RESULT', {}).get('CODE')
        message = body['rentBikeStatus'].get('RESULT', {}).get('MESSAGE')
    else:
        returned_rows = 0
        first_station = None
        code = body.get('CODE')
        message = body.get('MESSAGE')

    filter_rows.append({
        'url': url,
        'http_status': r.status_code,
        'result_code': code,
        'result_message': message,
        'returned_rows': returned_rows,
        'first_station_name': first_station,
    })

filter_test_df = pd.DataFrame(filter_rows)
filter_test_df.to_csv(OUTPUT_DIR / 'ddri_seoul_bike_stationid_filter_tests.csv', index=False, encoding='utf-8-sig')
print(filter_test_df.to_string(index=False))
print()
print('저장:', OUTPUT_DIR / 'ddri_seoul_bike_stationid_filter_tests.csv')


                                                                                               url  http_status result_code  result_message  returned_rows first_station_name
             http://openapi.seoul.go.kr:8088/6776635665636865343076784d5562/json/bikeList/1/5/2312          200    INFO-200 해당하는 데이터가 없습니다.              0               None
   http://openapi.seoul.go.kr:8088/6776635665636865343076784d5562/json/bikeList/1/5/stationId/2312          200    INFO-200 해당하는 데이터가 없습니다.              0               None
  http://openapi.seoul.go.kr:8088/6776635665636865343076784d5562/json/bikeList/1/5/?stationId=2312          200    INFO-000     정상 처리되었습니다.              5    102. 망원역 1번출구 앞
http://openapi.seoul.go.kr:8088/6776635665636865343076784d5562/json/bikeList/1/5/?stationId=ST-795          200    INFO-000     정상 처리되었습니다.              5    102. 망원역 1번출구 앞

저장: /Users/cheng80/Desktop/ddri_work/cheng80/api_output/ddri_seoul_bike_stationid_filter_tests.csv


## 4. 대표 15개 스테이션 매칭 검증

In [5]:

REP15_CSV = BASE_DIR / '3조 공유폴더' / '대표대여소_예측데이터_15개' / 'raw_data' / 'ddri_prediction_long_test_2025.csv'
rep15_df = pd.read_csv(REP15_CSV, encoding='utf-8-sig')
rep15_stations = rep15_df[['station_id', 'station_name']].drop_duplicates().sort_values('station_id').copy()
rep15_stations['station_id'] = rep15_stations['station_id'].astype(str)

rep15_matched = rep15_stations.merge(
    api_df[['station_num', 'stationId', 'stationName', 'parkingBikeTotCnt', 'rackTotCnt', 'stationLatitude', 'stationLongitude']],
    left_on='station_id',
    right_on='station_num',
    how='left',
)
rep15_matched['matched'] = rep15_matched['stationId'].notna()
rep15_matched.to_csv(OUTPUT_DIR / 'ddri_rep15_station_api_validation_official.csv', index=False, encoding='utf-8-sig')

print('매칭 성공:', int(rep15_matched['matched'].sum()), '/', len(rep15_matched))
print('매칭 실패 station_id:', rep15_matched.loc[~rep15_matched['matched'], 'station_id'].tolist())
print(rep15_matched[['station_id', 'station_name', 'stationId', 'stationName', 'parkingBikeTotCnt', 'rackTotCnt', 'matched']].to_string(index=False))
print()
print('저장:', OUTPUT_DIR / 'ddri_rep15_station_api_validation_official.csv')


매칭 성공: 14 / 15
매칭 실패 station_id: ['2323']
station_id               station_name stationId                     stationName parkingBikeTotCnt rackTotCnt  matched
      2312               청담역 13번 출구 앞    ST-795              2312. 청담역 13번 출구 앞                 2         10     True
      2320               도곡역 대치지구대 방향    ST-803              2320. 도곡역 대치지구대 방향                 8         10     True
      2321                   학여울역 사거리    ST-804                  2321. 학여울역 사거리                 1         10     True
      2323              주식회사 오뚜기 정문 앞       NaN                             NaN               NaN        NaN    False
      2328  르네상스 호텔 사거리 역삼지하보도 7번출구 앞    ST-811 2328. 르네상스 호텔 사거리 역삼지하보도 7번출구 앞                 1         10     True
      2348               포스코사거리(기업은행)    ST-797              2348. 포스코사거리(기업은행)                 3         23     True
      2354                   청담역 2번출구    ST-953                  2354. 청담역 2번출구                 1          5     True
      2359    

## 5. 현재 서비스 범위 161개 스테이션 매칭 검증

In [6]:

FULL_TEST_CSV = BASE_DIR / '3조 공유폴더' / '군집별 데이터_전체 스테이션' / 'full_data' / 'ddri_prediction_long_test_2025.csv'
full_test_df = pd.read_csv(FULL_TEST_CSV, encoding='utf-8-sig')
full_stations = full_test_df[['station_id']].drop_duplicates().sort_values('station_id').copy()
full_stations['station_id'] = full_stations['station_id'].astype(str)

full_matched = full_stations.merge(
    api_df[['station_num', 'stationId', 'stationName', 'parkingBikeTotCnt', 'rackTotCnt']],
    left_on='station_id',
    right_on='station_num',
    how='left',
)
full_matched['matched'] = full_matched['stationId'].notna()
full_matched.to_csv(OUTPUT_DIR / 'ddri_full161_station_api_validation_official.csv', index=False, encoding='utf-8-sig')

print('매칭 성공:', int(full_matched['matched'].sum()), '/', len(full_matched))
print('매칭 실패 station_id:', full_matched.loc[~full_matched['matched'], 'station_id'].tolist())
print()
print('저장:', OUTPUT_DIR / 'ddri_full161_station_api_validation_official.csv')


매칭 성공: 158 / 161
매칭 실패 station_id: ['2314', '2323', '3628']

저장: /Users/cheng80/Desktop/ddri_work/cheng80/api_output/ddri_full161_station_api_validation_official.csv


## 6. 공식 API 스키마와 서비스 연동 입력 확인

In [7]:

schema_mapping_df = pd.DataFrame([
    {'service_field': 'station_id_numeric', 'api_field': 'stationName prefix', 'note': '예: `2312. 청담역 13번 출구 앞`에서 숫자 prefix 추출 필요'},
    {'service_field': 'station_id_api', 'api_field': 'stationId', 'note': '예: `ST-795`; API 내부 식별자'},
    {'service_field': 'current_bike_stock', 'api_field': 'parkingBikeTotCnt', 'note': '현재 주차 자전거 수'},
    {'service_field': 'capacity', 'api_field': 'rackTotCnt', 'note': '거치대 개수'},
    {'service_field': 'lat', 'api_field': 'stationLatitude', 'note': '위도'},
    {'service_field': 'lon', 'api_field': 'stationLongitude', 'note': '경도'},
    {'service_field': 'utilization_reference', 'api_field': 'shared', 'note': '거치율 참고값'},
])
print(schema_mapping_df.to_string(index=False))

readiness_checks = pd.DataFrame([
    {'check_item': '공식 서비스명 bikeList 확인', 'status': '완료', 'detail': '명세서와 실호출 일치'},
    {'check_item': '1000건 페이지네이션 확인', 'status': '완료', 'detail': '1~1000, 1001~2000, 2001~3000 응답 확인; 3001~4000은 INFO-200'},
    {'check_item': '대표 15개 매칭', 'status': '부분완료', 'detail': f"{int(rep15_matched['matched'].sum())}/15 매칭"},
    {'check_item': '전체 161개 매칭', 'status': '부분완료', 'detail': f"{int(full_matched['matched'].sum())}/161 매칭"},
    {'check_item': 'stationId 필터 실동작', 'status': '미확인', 'detail': '시도한 호출 형식에서는 필터 미동작'},
    {'check_item': '잔여 자전거 계산 연동 준비', 'status': '조건부 가능', 'detail': 'numeric station_id ↔ API stationId 매핑 테이블 필요'},
])
print()
print('[검증 상태 요약]')
print(readiness_checks.to_string(index=False))


        service_field          api_field                                      note
   station_id_numeric stationName prefix 예: `2312. 청담역 13번 출구 앞`에서 숫자 prefix 추출 필요
       station_id_api          stationId                   예: `ST-795`; API 내부 식별자
   current_bike_stock  parkingBikeTotCnt                               현재 주차 자전거 수
             capacity         rackTotCnt                                    거치대 개수
                  lat    stationLatitude                                        위도
                  lon   stationLongitude                                        경도
utilization_reference             shared                                   거치율 참고값

[검증 상태 요약]
         check_item status                                                  detail
공식 서비스명 bikeList 확인     완료                                             명세서와 실호출 일치
    1000건 페이지네이션 확인     완료 1~1000, 1001~2000, 2001~3000 응답 확인; 3001~4000은 INFO-200
          대표 15개 매칭   부분완료                                                1

## 7. 대여소 마스터 API 대조와 누락 해석

- 활용 API:
  - `OA-21235 bikeStationMaster`
  - `tbCycleStationInfo`
- 목적:
  - 실시간 `bikeList`에서 누락된 스테이션이 대여소 마스터에는 존재하는지 확인
  - `numeric station_id`와 `ST-xxxx`를 더 안정적으로 연결할 수 있는지 확인


In [8]:
MASTER_KEY = '4e4d4147696368653930437364534e'
TARGET_UNMATCHED = ['2314', '2323', '3628']

def fetch_openapi_rows(api_key: str, service: str, start: int, end: int):
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/json/{service}/{start}/{end}/'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    payload = data.get(service) or data.get('stationInfo') or data.get('bikeStationMaster')
    if isinstance(payload, dict):
        return pd.DataFrame(payload.get('row', []))
    return pd.DataFrame()

station_info_pages = []
for start in [1, 1001, 2001, 3001, 4001]:
    df = fetch_openapi_rows(MASTER_KEY, 'tbCycleStationInfo', start, start + 999)
    if df.empty:
        break
    station_info_pages.append(df)

master_pages = []
for start in [1, 1001, 2001, 3001, 4001]:
    df = fetch_openapi_rows(MASTER_KEY, 'bikeStationMaster', start, start + 999)
    if df.empty:
        break
    master_pages.append(df)

station_info_df = pd.concat(station_info_pages, ignore_index=True)
master_df = pd.concat(master_pages, ignore_index=True)
station_info_df['RENT_NO_STR'] = station_info_df['RENT_NO'].astype(str).str.lstrip('0')

unmatched_station_info = station_info_df[station_info_df['RENT_NO_STR'].isin(TARGET_UNMATCHED)].copy()
unmatched_joined = unmatched_station_info.merge(
    master_df,
    left_on='RENT_ID',
    right_on='RNTLS_ID',
    how='left',
)

unmatched_joined.to_csv(OUTPUT_DIR / 'ddri_unmatched_station_master_crosscheck.csv', index=False, encoding='utf-8-sig')

print(unmatched_joined[[
    'RENT_NO_STR', 'RENT_ID', 'RENT_NM', 'RENT_ID_NM', 'HOLD_NUM', 'STA_ADD1', 'STA_ADD2', 'LAT', 'LOT'
]].to_string(index=False))
print()
print('저장:', OUTPUT_DIR / 'ddri_unmatched_station_master_crosscheck.csv')


RENT_NO_STR RENT_ID       RENT_NM          RENT_ID_NM HOLD_NUM                   STA_ADD1      STA_ADD2       LAT        LOT
       3628 ST-2689     역삼1동 주민센터     3628. 역삼1동 주민센터       10 서울특별시 강남구 역삼로7길 16 역삼1문화센터               37.495213 127.033028
       2314  ST-798       청담나들목입구       2314. 청담나들목입구       10        서울특별시 강남구 삼성동 111-4               37.521015 127.060150
       2323  ST-806 주식회사 오뚜기 정문 앞 2323. 주식회사 오뚜기 정문 앞       10         서울특별시 강남구 영동대로 308 주식회사 오뚜기 정문 앞 37.502213 127.067207

저장: /Users/cheng80/Desktop/ddri_work/cheng80/api_output/ddri_unmatched_station_master_crosscheck.csv


## 8. 운영 상태 해석 규칙 초안

현재 공식 API 조합을 기준으로, 서비스 연동 단계에서는 아래처럼 해석한다.

- `bikeList`에 존재하고 좌표가 정상값이면: `운영 중`
- `bikeList`에는 없지만 `tbCycleStationInfo` 또는 `bikeStationMaster`에는 존재하면: `실시간 비노출 / 비활성 후보`
- `bikeStationMaster` 좌표가 `(0, 0)`이면: `비정상 좌표 / 운영 중지 후보`
- `bikeList` 미매칭 또는 `(0, 0)` 좌표 대여소는 사용자 화면에서 `실시간 정보 없음` 또는 `예측 제외`로 처리한다.
- 운영자 화면에서는 위 대여소를 `예외 대여소 목록`으로 별도 표기한다.


## 9. 매핑 테이블 활용 방식

추가 호출과 반복 매핑을 줄이기 위해, 아래 lookup 파일을 서비스 전처리 단계에서 먼저 로드한다.

- `cheng80/api_output/ddri_station_id_api_lookup.csv`
- 핵심 컬럼:
  - `station_id`
  - `resolved_api_station_id`
  - `operational_status`

권장 흐름:

1. 예측 대상 `station_id` 목록 준비
2. lookup CSV와 조인해 `resolved_api_station_id` 확보
3. `operational_status != realtime_available` 대여소는 예외 처리
4. 실시간 API 전체 응답을 한 번 받아 `resolved_api_station_id` 기준으로 조인
5. 최종적으로 `current_bike_stock`, `capacity`, 예외 여부를 예측 결과에 붙인다.


In [9]:
LOOKUP_CSV = OUTPUT_DIR / 'ddri_station_id_api_lookup.csv'
lookup_df = pd.read_csv(LOOKUP_CSV, encoding='utf-8-sig')
lookup_df['station_id'] = lookup_df['station_id'].astype(str)

service_target_df = pd.read_csv(
    BASE_DIR / '3조 공유폴더' / '군집별 데이터_전체 스테이션' / 'full_data' / 'ddri_prediction_long_test_2025.csv',
    encoding='utf-8-sig',
)
service_target_df = service_target_df[['station_id']].drop_duplicates().copy()
service_target_df['station_id'] = service_target_df['station_id'].astype(str)

service_target_with_lookup = service_target_df.merge(lookup_df, on='station_id', how='left')

print('서비스 대상 station 수:', len(service_target_with_lookup))
print('lookup 매핑 성공 수:', service_target_with_lookup['resolved_api_station_id'].notna().sum())
print('예외 후보 수:', (service_target_with_lookup['operational_status'] != 'realtime_available').sum())
print()
print(service_target_with_lookup.head(10).to_string(index=False))


서비스 대상 station 수: 161
lookup 매핑 성공 수: 161
예외 후보 수: 3

station_id resolved_api_station_id operational_status mapping_source exception_note
      2301                  ST-777 realtime_available       bikeList            NaN
      2302                  ST-787 realtime_available       bikeList            NaN
      2303                  ST-788 realtime_available       bikeList            NaN
      2304                  ST-789 realtime_available       bikeList            NaN
      2305                  ST-790 realtime_available       bikeList            NaN
      2306                  ST-791 realtime_available       bikeList            NaN
      2307                  ST-792 realtime_available       bikeList            NaN
      2308                  ST-793 realtime_available       bikeList            NaN
      2309                  ST-783 realtime_available       bikeList            NaN
      2310                  ST-794 realtime_available       bikeList            NaN


In [10]:
realtime_join_ready_df = api_df.rename(columns={
    'stationId': 'resolved_api_station_id',
    'parkingBikeTotCnt': 'current_bike_stock',
    'rackTotCnt': 'current_capacity',
})[['resolved_api_station_id', 'stationName', 'current_bike_stock', 'current_capacity', 'shared']].copy()

service_realtime_df = service_target_with_lookup.merge(
    realtime_join_ready_df,
    on='resolved_api_station_id',
    how='left',
)

service_realtime_df['realtime_join_success'] = service_realtime_df['current_bike_stock'].notna()
service_realtime_df.to_csv(OUTPUT_DIR / 'ddri_service_realtime_join_preview.csv', index=False, encoding='utf-8-sig')

print(service_realtime_df.head(10).to_string(index=False))
print()
print('실시간 join 성공 수:', int(service_realtime_df['realtime_join_success'].sum()))
print('실시간 join 실패 수:', int((~service_realtime_df['realtime_join_success']).sum()))
print('실시간 join 실패 station_id:', service_realtime_df.loc[~service_realtime_df['realtime_join_success'], 'station_id'].tolist())
print()
print('저장:', OUTPUT_DIR / 'ddri_service_realtime_join_preview.csv')


station_id resolved_api_station_id operational_status mapping_source exception_note                    stationName current_bike_stock current_capacity shared  realtime_join_success
      2301                  ST-777 realtime_available       bikeList            NaN               2301. 현대고등학교 건너편                  6               10     60                   True
      2302                  ST-787 realtime_available       bikeList            NaN 2302. 교보타워 버스정류장(신논현역 3번출구 후면)                  5               10     50                   True
      2303                  ST-788 realtime_available       bikeList            NaN                2303. 논현역 10번출구                  0               15      0                   True
      2304                  ST-789 realtime_available       bikeList            NaN                   2304. 대현그린타워                  0               10      0                   True
      2305                  ST-790 realtime_available       bikeList            NaN            

## 10. 외부 요약 문서 재검증

`cheng80/02_ddri_api_verification_summary.md`에 적힌 핵심 주장들을 현재 노트북의 실검증 결과와 다시 대조한다.

목적:

- 외부 요약 문서에서 바로 재사용 가능한 정보와 수정이 필요한 정보를 구분
- 현재 프로젝트의 정본 판단을 노트북 기준으로 고정


In [11]:
MASTER_KEY = '4e4d4147696368653930437364534e'
revalidation_rows = []

revalidation_rows.append({
    'claim_item': '대표 15개 공식 API 매칭',
    'summary_claim': '15/15',
    'notebook_result': f"{int(rep15_matched['matched'].sum())}/15",
    'status': '일치' if int(rep15_matched['matched'].sum()) == 15 else '불일치',
    'note': '현재 bikeList 기준 2323 미매칭',
})

revalidation_rows.append({
    'claim_item': '전체 161개 공식 API 매칭',
    'summary_claim': '159/161',
    'notebook_result': f"{int(full_matched['matched'].sum())}/161",
    'status': '일치' if int(full_matched['matched'].sum()) == 159 else '불일치',
    'note': '현재 bikeList 기준 2314, 2323, 3628 미매칭',
})

bike_non_empty_pages = int((page_summary_df['returned_rows'] > 0).sum())
bike_total_rows = int(page_summary_df['returned_rows'].sum())
revalidation_rows.append({
    'claim_item': 'bikeList 전체 row 수 / 호출 수',
    'summary_claim': '3223개 / 4회 호출',
    'notebook_result': f"{bike_total_rows}개 / 데이터 응답 {bike_non_empty_pages}회",
    'status': '불일치',
    'note': 'bikeList의 list_total_count는 전체 건수로 보이지 않으며, 실응답은 1~3000 범위까지만 존재',
})

stationid_filter_failed = int((filter_test_df['returned_rows'] == 0).sum() >= 2 and (filter_test_df['first_station_name'] == '102. 망원역 1번출구 앞').sum() >= 1)
revalidation_rows.append({
    'claim_item': 'stationId 필터 사용 가능성',
    'summary_claim': '명시적 언급 없음',
    'notebook_result': '현재 검증한 호출 형식에서는 미동작',
    'status': '추가 보완',
    'note': 'querystring stationId도 실제 필터링되지 않았음',
})

def test_key_compat(api_key, service):
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/json/{service}/1/5/'
    try:
        r = requests.get(url, timeout=20)
        d = r.json()
        if service in d or 'rentBikeStatus' in d or 'stationInfo' in d:
            return 'OK'
        result = d.get('RESULT', {}) if isinstance(d, dict) else {}
        code = result.get('CODE') or d.get('CODE')
        return code or 'UNKNOWN'
    except Exception as e:
        return f'ERR:{type(e).__name__}'

key_tests = []
for key_name, key_value in [('bike', SEOUL_BIKE_KEY), ('rtd', os.environ.get('SEOUL_RTD_API_KEY')), ('master', MASTER_KEY)]:
    for service in ['bikeList', 'tbCycleStationInfo']:
        if key_value:
            key_tests.append({'key_name': key_name, 'service': service, 'result': test_key_compat(key_value, service)})
key_tests_df = pd.DataFrame(key_tests)

three_keys_ok = key_tests_df['result'].eq('OK').all() if not key_tests_df.empty else False
revalidation_rows.append({
    'claim_item': '3개 키 bikeList/tbCycleStationInfo 호환성',
    'summary_claim': '3개 키 모두 OK',
    'notebook_result': ' / '.join([f"{r.key_name}:{r.service}={r.result}" for r in key_tests_df.itertuples()]),
    'status': '일치' if three_keys_ok else '부분확인',
    'note': '단순 1~5 호출 기준 확인이며 서비스 정책 고정 보장은 아님',
})

revalidation_df = pd.DataFrame(revalidation_rows)
revalidation_df.to_csv(OUTPUT_DIR / 'ddri_api_summary_revalidation_checklist.csv', index=False, encoding='utf-8-sig')
key_tests_df.to_csv(OUTPUT_DIR / 'ddri_api_key_compatibility_check.csv', index=False, encoding='utf-8-sig')
print(revalidation_df.to_string(index=False))
print()
print('[key compatibility]')
print(key_tests_df.to_string(index=False))
print()
print('저장:', OUTPUT_DIR / 'ddri_api_summary_revalidation_checklist.csv')
print('저장:', OUTPUT_DIR / 'ddri_api_key_compatibility_check.csv')


                          claim_item summary_claim                                                                                                                                 notebook_result status                                                             note
                    대표 15개 공식 API 매칭         15/15                                                                                                                                           14/15    불일치                                          현재 bikeList 기준 2323 미매칭
                   전체 161개 공식 API 매칭       159/161                                                                                                                                         158/161    불일치                              현재 bikeList 기준 2314, 2323, 3628 미매칭
            bikeList 전체 row 수 / 호출 수 3223개 / 4회 호출                                                                                                                               2684개 / 데이터 응답 3회    불

## 11. 현재 결론


- 공식 실시간 API는 `bikeList` 기준으로 정상 호출된다.
- 현재 응답은 `3000`건 범위 안에서 조회되며, 서비스 적용 시 `1000건` 단위 반복 호출 로직이 필요하다.
- `stationId` 필터는 명세서에 적혀 있지만, 이번 검증에서 시도한 URL 형식으로는 동작하지 않았다.
- 따라서 현재 기준으로는 `서울 전체 페이지 조회 -> 강남/대상 station_id 로컬 필터링` 구조가 가장 안전하다.
- 대표 15개 기준 `14/15`, 전체 161개 기준 `158/161`만 매칭되며, 누락 스테이션은 `2314`, `2323`, `3628`이다.
- 위 3개는 `bikeList`에는 없지만 `tbCycleStationInfo`와 `bikeStationMaster`에는 존재하므로, 현재는 `실시간 비노출 / 비활성 후보`로 해석한다.
- 마스터 API 좌표가 `(0, 0)`인 경우도 같은 수준의 예외 후보로 보고, 운영자 화면에서 별도 관리한다.
- 서비스 후처리로 넘어가려면 `stationName` 숫자 prefix를 이용한 `numeric station_id ↔ stationId(ST-xxxx)` 매핑 테이블을 정식 산출물로 고정해야 한다.
